## Tactical Development


### Dataset :: (fake dataset - databricks market place)



- "databricks_simulated_retail_customer_data"

1) The first step is to explore the dataset as much as possible, to identify all the possible relationships existent between the tables and volumes;

2) Ideally more than one architecture for data modeling, until "best" pipeline and insight is found;


In [0]:
# %sql
# Sample Script 
# CREATE SCHEMA IF NOT EXISTS medallion_catalog.db_bronze
# MANAGED LOCATION 'abfss://catalog@container.dfs.core.windows.net/folder';

# USE CATALOG medallion_catalog;
# USE SCHEMA db_bronze;

# DROP TABLE IF EXISTS medallion_catalog.db_bronze.companies;

# CREATE TABLE IF NOT EXISTS medallion_catalog.db_bronze.companies 
# AS
# SELECT company_name, founded_date, country
#   FROM read_files('/Volumes/medallion_catalog/db_landing/vol_medallion/top_tech_companies/',
#   format => 'csv',
#   header => true);


### Volumes

- (dealing with streams)

In [0]:
# # Import necessary functions
# from pyspark.sql.functions import input_file_name, current_timestamp

# # Configuration
# source_path = "/mnt/raw-data/incoming/"
# checkpoint_path = "/mnt/checkpoints/autoloader_drift/"
# schema_location = "/mnt/schemas/autoloader_drift/"
# target_table = "bronze_sensor_data"

# # 1. Define the Auto Loader Stream
# df_stream = (spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format", "json")             # Source format (json, csv, parquet, etc.)
#     .option("cloudFiles.schemaLocation", schema_location)
#     .option("cloudFiles.inferColumnTypes", "true")   # Better type inference
#     .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Automatic drift handling
#     .load(source_path)
#     .withColumn("source_file", input_file_name())    # Track source for audit
#     .withColumn("ingested_at", current_timestamp())
# )

# # 2. Write to Delta Table
# query = (df_stream.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("checkpointLocation", checkpoint_path)
#     .option("mergeSchema", "true")                  # Allow target table to evolve
#     .trigger(availableNow=True)                    # Process all new data and stop
#     .toTable(target_table)
# )

# query.awaitTermination()

#### retail-pipeline (customers)

2. Auto Loader

**Advantages:**

**Scalability:** Efficiently scales to discover and ingest millions or billions of files using cloud-native APIs, reducing discovery costs.

**Schema Handling:** Robust automatic schema inference and evolution support.

**Operational Ease:** Built on Spark Structured Streaming, it automatically tracks progress via checkpointing, removing the need for manual orchestration.

**Disadvantages:**

**Complexity:** Slightly more complex to set up and manage compared to COPY INTO (requires checkpoint locations, specific configurations).

**Reprocessing:** Harder to reprocess a select subset of files once ingested.

**Best Cases for Usage:** Continuous or near real-time incremental ingestion of files from cloud storage.
Handling data with frequently evolving schemas (JSON, semi-structured data). Any file-based ingestion where high volume and production reliability are key concerns.

**Overview ::**

(Python, Scala, SQL (cloudFiles format)):
Near real-time/Continuous (micro-batch or continuous mode)
Cloud object storage files	
Efficient, scalable file notification or directory listing
Automatic inference and evolution handling	
Manual compute management (runs on a cluster)




##### readStream

In [0]:
# Import necessary functions
from pyspark.sql.functions import input_file_name, current_timestamp

# Configuration
source_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/customer_auto/"  #only one for now
checkpoint_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/customer_auto/_checkpoint_stream"
schema_location = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/customer_auto/"
target_table = "end_to_end_retail.db_landing.customer_autoloader"

#### Simple example for one file only
- add other later

In [0]:
customers_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "timestamp DATE")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Automatic drift handling
    .load(source_path)
    
    # .option("cloudFiles.useNotifications", "true")
    # .useNotifications will read from queues message coming from cloud services
    # AWS Event Notification, Azure Event Grid
    # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

customers_transformed_df = (
    customers_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
# 2. Write to Delta Table
query = (customers_transformed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")                  # Allow target table to evolve
    .trigger(availableNow=True)                    # Process all new data and stop
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
query.stop()

##### readStream tracking columns

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.customer_autoloader limit 5

#### retail-pipeline (orders)

In [0]:
# Import necessary functions
from pyspark.sql.functions import input_file_name, current_timestamp

# Configuration
source_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_orders/"  #only one for now
checkpoint_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_orders/_checkpoint_stream"
schema_location = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_orders/"
target_table = "end_to_end_retail.db_landing.retail_pipeline_autoloader"

In [0]:
retail_orders_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "order_timestamp DATE")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Automatic drift handling
    .load(source_path)
    
    # .option("cloudFiles.useNotifications", "true")
    # .useNotifications will read from queues message coming from cloud services
    # AWS Event Notification, Azure Event Grid
    # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

retail_orders_transformed_df = (
    retail_orders_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
# 2. Write to Delta Table
query = (retail_orders_transformed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")                  # Allow target table to evolve
    .trigger(availableNow=True)                    # Process all new data and stop
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
query.stop()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.retail_pipeline_autoloader limit 5

In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json/00.json")
display(df.head(5))

#### retail-pipeline (status)

In [0]:
# Import necessary functions
from pyspark.sql.functions import input_file_name, current_timestamp

# Configuration
source_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_status/"  #only one for now
checkpoint_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_status/_checkpoint_stream"
schema_location = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_status/"
target_table = "end_to_end_retail.db_landing.retail_pipeline_status_autoloader"

In [0]:
retail_status_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    # .option("cloudFiles.schemaHints", "status_timestamp DATE")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Automatic drift handling
    .load(source_path)
    
    # .option("cloudFiles.useNotifications", "true")
    # .useNotifications will read from queues message coming from cloud services
    # AWS Event Notification, Azure Event Grid
    # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

retail_status_transformed_df = (
    retail_status_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
# 2. Write to Delta Table
query = (retail_status_transformed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")                  # Allow target table to evolve
    .trigger(availableNow=True)                    # Process all new data and stop
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
query.stop()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.retail_pipeline_status_autoloader limit 5


In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/status/stream_json/00.json")
display(df.head(5))